# 01 Own cohort: WT 12 mo (Figure 2) and young references WT 3 mo, AQP4 KO 3 mo (Supplementary Figure S4)
Run K0 to K7 in order in a fresh kernel (K6 = Supplementary Figure S4, K8 = Figure 2, K7 = Supplementary Table S2). Figures go to `figures/panels/` and `final_data/figures/`, source data to `final_data/supplementary_tables/Supplementary_Table_S2_source_data_Figure2_FigureS4.xlsx`.

In [ ]:
# ==========================================================
# K0) Setup: paths, style, helpers (own cohort, 247-gene panel)
# ==========================================================
import gc, re, warnings
import numpy as np, pandas as pd
import anndata as ad, scanpy as sc
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
import statsmodels.api as sm, statsmodels.formula.api as smf
from statsmodels.stats.contingency_tables import StratifiedTable
from statsmodels.stats.multitest import multipletests
from scipy.spatial import cKDTree
from pathlib import Path
from PIL import Image
warnings.filterwarnings("ignore")

import os
# Project folder with data/, tables/, figures/ and final_data/ (see README).
# Default: the parent of the notebooks folder. Override with the environment variable SPP1_PROJECT_DIR.
OUT = Path(os.environ.get("SPP1_PROJECT_DIR", Path.cwd().parent)).resolve()
FIG, TAB, DAT = OUT / "figures", OUT / "tables", OUT / "data"
PANELS = FIG / "panels"; PANELS.mkdir(parents=True, exist_ok=True)
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5,
    "legend.fontsize": 6.5, "axes.linewidth": 0.6, "pdf.fonttype": 42, "svg.fonttype": "none",
    "savefig.dpi": 300, "figure.dpi": 150, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False})
MM = 1 / 25.4

def dense(M):
    return M.toarray() if hasattr(M, "toarray") else np.asarray(M)

FINAL = OUT / "final_data" / "figures"; FINAL.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    """Panels and drafts go to figures/panels; Fig1 to Fig3 additionally to final_data/figures."""
    for d in [PANELS] + ([FINAL] if re.fullmatch(r"Fig(\d|S\d)", name) else []):
        fig.savefig(d / f"{name}.pdf", bbox_inches="tight")
        fig.savefig(d / f"{name}.png", dpi=300, bbox_inches="tight")
        if d == FINAL:                                   # Frontiers: TIFF, RGB, 300 dpi, LZW
            Image.open(d / f"{name}.png").convert("RGB").save(d / f"{name}.tif", compression="tiff_lzw", dpi=(300, 300))
    plt.show()

# shared colours (identical to the 10x figures)
C_MG, C_POS, C_NEG, C_AS = "#e0a100", "#c2185b", "#f1d48a", "#3b5bdb"
C_FIB, C_BOR, C_BG = "#9fb3c8", "#d3dce6", "#efefef"
C_YOUNG, C_OLD = "#a9b8ea", "#3b5bdb"
CTX = ["Grey", "Border", "Fibre"]

In [ ]:
# ==========================================================
# K1) Load our labelled sections (SPF 12 wk, AQP4 KO 12 wk, healthy 1 y), counts + coordinates
# ==========================================================
SAMPLES = {"SPF_Coronar": ("SPF 12 wk", "young", 2.8),
           "AQP4KO_sagital": ("AQP4 KO 12 wk", "young_KO", 2.8),
           "Thy_CTRL": ("Healthy 1 y", "aged", 12.0)}
ref = sc.read_h5ad(DAT / "own_cohort_xenium_247panel.h5ad", backed="r")
print("obs columns:", ref.obs.columns.tolist()); print("obsm:", list(ref.obsm.keys()))
smp_col = [c for c in ["sample", "sample_name", "Sample"] if c in ref.obs.columns][0]
keep = ref.obs[smp_col].astype(str).isin(SAMPLES).to_numpy()
k1 = ref[keep].to_memory(); del ref; gc.collect()
k1.obs["sample"] = k1.obs[smp_col].astype(str)
k1.obs["label"] = k1.obs["sample"].map(lambda s: SAMPLES[s][0])
k1.obs["group"] = k1.obs["sample"].map(lambda s: SAMPLES[s][1])
k1.obs["age_mo"] = k1.obs["sample"].map(lambda s: SAMPLES[s][2]).astype(float)
k1.obs["cell_type"] = k1.obs["cell_type"].astype(str)
if "spatial" in k1.obsm:
    xy = np.asarray(k1.obsm["spatial"])[:, :2]
else:
    xy = k1.obs[["x_centroid", "y_centroid"]].to_numpy()
k1.obs["x"], k1.obs["y"] = xy[:, 0], xy[:, 1]
Cnt = k1.layers["counts"] if "counts" in k1.layers else k1.X
k1.obs["Spp1_n"] = dense(Cnt[:, list(k1.var_names).index("Spp1")]).ravel()
print(k1)
print(pd.crosstab(k1.obs["cell_type"], k1.obs["label"]).to_string())
print("section extent (µm):\n", k1.obs.groupby("label")[["x", "y"]].agg(lambda v: v.max() - v.min()).round(0))

In [ ]:
# ==========================================================
# K2) Definitions as for the 10x data: microglia, astrocytes, local fibre context (k = 30), astrocytes within 25 µm
# ==========================================================
o = k1.obs
o["is_mg"] = o["cell_type"] == "Microglia"
o["is_astro"] = o["cell_type"] == "Astrocyte"
K = 30; o["f_oligo"] = np.nan
for s in o["sample"].unique():
    m = np.where((o["sample"] == s).to_numpy())[0]; pts = o[["x", "y"]].to_numpy()[m]
    _, nn = cKDTree(pts).query(pts, k=K + 1)
    o.iloc[m, o.columns.get_loc("f_oligo")] = (o["cell_type"].to_numpy()[m][nn[:, 1:]] == "Oligodendrocyte").mean(1)
o["ctx_bin"] = pd.cut(o["f_oligo"], [-0.01, 0.2, 0.4, 1.0], labels=CTX).astype(str)

# astrocytes within 25 µm of every microglia
o["n_astro25"] = np.nan
for s in o["sample"].unique():
    m = np.where((o["sample"] == s).to_numpy())[0]; pts = o[["x", "y"]].to_numpy()[m]
    im, ia = o["is_mg"].to_numpy()[m], o["is_astro"].to_numpy()[m]
    o.iloc[m[im], o.columns.get_loc("n_astro25")] = cKDTree(pts[ia]).query_ball_point(pts[im], 25, return_length=True)

# Spp1 by cell type and sample
grp = o["cell_type"].where(o["cell_type"].isin(["Microglia", "Astrocyte", "Oligodendrocyte", "OPC", "Endothelial",
                                                 "Mural", "VLMC / fibroblast"]), "Neurons / other")
t3 = (pd.DataFrame({"ct": grp, "label": o["label"], "pos": o["Spp1_n"] >= 1, "n": o["Spp1_n"]})
        .groupby(["ct", "label"]).agg(pct=("pos", "mean"), mean_pos=("n", lambda v: v[v >= 1].mean() if (v >= 1).any() else np.nan),
                                      n_cells=("pos", "size")).reset_index())
t3["pct"] *= 100
t3.to_csv(TAB / "K1_03_spp1_by_celltype.csv", index=False)
print(t3.pivot(index="ct", columns="label", values="pct").round(1).to_string())
print(t3.pivot(index="ct", columns="label", values="mean_pos").round(2).to_string())
print(pd.crosstab(o.loc[o["is_mg"], "ctx_bin"], o.loc[o["is_mg"], "label"]).to_string())

In [ ]:
# ==========================================================
# K3) Home region: fibre and ventricle cells inherit the majority grey-matter region of their neighbours
# ==========================================================
o_all = k1.obs
REGCOL = "region_coarse"
NONHOME = {"White matter", "Ventricle / CP", "nan", "None"}
o_all["home_region"] = o_all[REGCOL].astype(str)
for s in o_all["sample"].unique():
    m = np.where((o_all["sample"] == s).to_numpy())[0]
    pts = o_all[["x", "y"]].to_numpy()[m]; reg = o_all[REGCOL].astype(str).to_numpy()[m]
    need = np.where(np.isin(reg, list(NONHOME)))[0]
    tree = cKDTree(pts); home = reg.copy()
    for kk in [100, 300, 1000]:                                   # widen until a grey-matter neighbour is found
        if len(need) == 0: break
        _, nn = tree.query(pts[need], k=min(kk, len(pts)))
        still = []
        for i_, row in zip(need, nn):
            lab = reg[row]; lab = lab[~np.isin(lab, list(NONHOME))]
            if len(lab): home[i_] = pd.Series(lab).mode().iloc[0]
            else: still.append(i_)
        need = np.array(still, int)
    o_all.iloc[m, o_all.columns.get_loc("home_region")] = home

cnt = pd.crosstab(o_all["home_region"], o_all["sample"])
SHARED = sorted(r for r in cnt.index[(cnt["SPF_Coronar"] >= 500) & (cnt["Thy_CTRL"] >= 500)]
                if r not in {"Cerebellum", "Hindbrain", "Midbrain"} | NONHOME)
o_all["shared"] = o_all["home_region"].isin(SHARED)
print("shared home regions:", SHARED)
pos_all = o_all["is_mg"] & (o_all["Spp1_n"] >= 1)
print("\nSpp1+ microglia by home region (analysed = shared):")
print(pd.crosstab([o_all.loc[pos_all, "home_region"]], o_all.loc[pos_all, "label"]).to_string())
print("\nshare of cells kept:", o_all.groupby("label")["shared"].mean().round(2).to_dict())
o = o_all[o_all["shared"]].copy()

In [ ]:
# ==========================================================
# K4) Core statistics: age effect (two young references), clustering, astrocyte association in the aged animal
# ==========================================================
o_all = k1.obs
assert all(c in o_all for c in ["home_region", "shared", "ctx_bin", "n_astro25"]), "run K2 and K3 first"
mg_all = o_all[o_all["is_mg"]].copy()
mg_all["spp1"] = (mg_all["Spp1_n"] >= 1).astype(int)
mg_all["aged"] = (mg_all["group"] == "aged").astype(int)
mg_all["stratum"] = mg_all["ctx_bin"] + "|" + mg_all["home_region"]
R = {}

def glm_or(df, formula, term):
    f = smf.glm(formula, data=df, family=sm.families.Binomial()).fit(cov_type="HC1")
    lo, hi = np.exp(f.conf_int().loc[term]); return np.exp(f.params[term]), lo, hi, f.pvalues[term]

def ctx_table(df, title):
    t = df.groupby(["ctx_bin", "label"]).agg(n=("spp1", "size"), k=("spp1", "sum")); t["pct"] = 100 * t["k"] / t["n"]
    print(f"\n{title}  (% Spp1+, then k):"); print(t["pct"].unstack().round(1).reindex(CTX).to_string())
    print(t["k"].unstack().reindex(CTX).to_string()); return t

# --- A: forebrain (shared territory), aged vs SPF
A = mg_all[mg_all["shared"] & mg_all["sample"].isin(["SPF_Coronar", "Thy_CTRL"])]
R["tabA"] = ctx_table(A, "A  forebrain, aged vs SPF")
R["A"] = glm_or(A, "spp1 ~ aged + ctx_bin", "aged")
R["A_reg"] = glm_or(A, "spp1 ~ aged + ctx_bin + home_region", "aged")
print("   OR (context): {:.2f} [{:.2f}, {:.2f}], p = {:.2g}".format(*R["A"]))
print("   OR (context + home region): {:.2f} [{:.2f}, {:.2f}], p = {:.2g}".format(*R["A_reg"]))

# --- B: same section plane, aged vs AQP4 KO, whole section (home regions present in both)
B = mg_all[mg_all["sample"].isin(["AQP4KO_sagital", "Thy_CTRL"])]
both = B.groupby("home_region")["sample"].nunique().loc[lambda v: v == 2].index
B = B[B["home_region"].isin(both)]
R["tabB"] = ctx_table(B, "B  same plane, aged vs AQP4 KO (whole section)")
R["B"] = glm_or(B, "spp1 ~ aged + ctx_bin", "aged")
R["B_reg"] = glm_or(B, "spp1 ~ aged + ctx_bin + home_region", "aged")
print("   OR (context): {:.2f} [{:.2f}, {:.2f}], p = {:.2g}".format(*R["B"]))
print("   OR (context + home region): {:.2f} [{:.2f}, {:.2f}], p = {:.2g}".format(*R["B_reg"]))
R["regB"] = B.groupby(["home_region", "label"])["spp1"].agg(["sum", "size"]).unstack()

# --- C: clustering per animal (permutation within context x home region)
rng_ = np.random.default_rng(0); R["clus"] = {}
for s, sel in [("SPF_Coronar", mg_all["sample"] == "SPF_Coronar"), ("AQP4KO_sagital", mg_all["sample"] == "AQP4KO_sagital"),
               ("Thy_CTRL", mg_all["sample"] == "Thy_CTRL")]:
    ta = mg_all[sel]; pts = ta[["x", "y"]].to_numpy(); lab = ta["spp1"].to_numpy(); strat = ta["stratum"].to_numpy()
    pairs = cKDTree(pts).query_ball_point(pts, 50)
    def clust(l):
        p = np.where(l == 1)[0]; return np.mean([any(l[j] == 1 for j in pairs[i] if j != i) for i in p])
    obs_c = clust(lab); null = []
    for _ in range(1000):
        l = lab.copy()
        for b in np.unique(strat):
            ib = np.where(strat == b)[0]; l[ib] = rng_.permutation(l[ib])
        null.append(clust(l))
    null = np.array(null); R["clus"][s] = (obs_c, null, (np.sum(null >= obs_c) + 1) / 1001)
    print(f"C  clustering {SAMPLES[s][0]}: {100 * obs_c:.0f}% vs null {100 * null.mean():.0f}% "
          f"(p = {R['clus'][s][2]:.3f}, n Spp1+ = {lab.sum()})")

# --- D: astrocyte coverage within the aged animal (whole section), context + home region
D = mg_all[mg_all["sample"] == "Thy_CTRL"]
R["D"] = glm_or(D, "spp1 ~ n_astro25 + ctx_bin + home_region", "n_astro25")
tabs = [pd.crosstab(tt["spp1"], tt["n_astro25"] >= 1).reindex(index=[1, 0], columns=[True, False], fill_value=0).to_numpy()
        for _, tt in D.groupby("stratum") if tt["spp1"].sum() > 0 and (tt["spp1"] == 0).sum() > 0]
stt = StratifiedTable(tabs); R["D_cmh"] = (stt.oddsratio_pooled, stt.test_null_odds().pvalue)
R["D_fore"] = glm_or(D[D["shared"]], "spp1 ~ n_astro25 + ctx_bin", "n_astro25")
print("\nD  aged, per astrocyte neighbour (whole section): OR {:.2f} [{:.2f}, {:.2f}], p = {:.3f}".format(*R["D"]))
print(f"   CMH (any contact, strata = context x home region): OR {R['D_cmh'][0]:.2f}, p = {R['D_cmh'][1]:.3f}")
print("   forebrain only: OR {:.2f} [{:.2f}, {:.2f}], p = {:.3f}".format(*R["D_fore"]))
print(D.groupby(["ctx_bin", "spp1"])["n_astro25"].agg(["mean", "size"]).round(2).unstack().to_string())

mg_all.to_csv(TAB / "K1_04_microglia_table.csv")

In [ ]:
# ==========================================================
# K5) Matched forebrain: home regions present in all three sections
# ==========================================================
cnt3 = pd.crosstab(o_all["home_region"], o_all["sample"])
FORE = [r for r in cnt3.index if (cnt3.loc[r] >= 300).all() and r not in {"Hindbrain", "Midbrain", "Cerebellum"}]
print("matched forebrain regions:", FORE)
M = mg_all[mg_all["home_region"].isin(FORE)]
t = M.groupby(["ctx_bin", "label"]).agg(n=("spp1", "size"), k=("spp1", "sum")); t["pct"] = 100 * t["k"] / t["n"]
print(t["pct"].unstack().round(1).reindex(CTX).to_string()); print(t["k"].unstack().reindex(CTX).to_string())
for ref in ["SPF_Coronar", "AQP4KO_sagital"]:
    d = M[M["sample"].isin([ref, "Thy_CTRL"])]
    print(f"aged vs {SAMPLES[ref][0]} (matched forebrain), context + region:",
          "OR {:.2f} [{:.2f}, {:.2f}], p = {:.2g}".format(*glm_or(d, "spp1 ~ aged + ctx_bin + home_region", "aged")))
print(M.groupby(["home_region", "label"])["spp1"].agg(["sum", "size"]).unstack().to_string())
R["FORE"] = FORE

In [ ]:
# ==========================================================
# K6a) Orientation check: each section in four rotations, pick the right one for ORIENT
# ==========================================================
fig, axes = plt.subplots(3, 4, figsize=(180 * MM, 140 * MM))
for r, s in enumerate(["SPF_Coronar", "AQP4KO_sagital", "Thy_CTRL"]):
    oo = o_all[o_all["sample"] == s]
    for c, rot in enumerate([0, 90, 180, 270]):
        t = np.deg2rad(rot); x, y = oo["x"] - oo["x"].mean(), oo["y"] - oo["y"].mean()
        xr, yr = x * np.cos(t) + y * np.sin(t), -x * np.sin(t) + y * np.cos(t)
        ax = axes[r, c]; ax.scatter(xr, yr, s=0.05, c=np.where(oo["cell_type"] == "Oligodendrocyte", "#3b5bdb", "#cccccc"), lw=0, rasterized=True)
        ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off"); ax.set_title(f"{s}  rot {rot}", fontsize=6)
fig.suptitle("# K6a  orientation check (blue = oligodendrocytes, white matter)", fontsize=8, x=0.01, ha="left")
plt.show()

# rostral should point left and dorsal up (sagittal), dorsal up (coronal); add True for a left-right or up-down flip
ORIENT = {"SPF_Coronar": (0, False, False), "AQP4KO_sagital": (180, False, False), "Thy_CTRL": (270, False, False)}

In [ ]:
# ==========================================================
# K6) Supplementary Figure S4: young reference mice (WT 3 mo, AQP4 KO 3 mo) and WT 12 mo
# ==========================================================
from matplotlib.patches import Circle, Ellipse, FancyBboxPatch, Patch
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportion_confint

# display labels (SAMPLES keeps the original labels, which are used to look up the Z2 table in panel H)
LBL = {"SPF_Coronar": "WT\n3 mo", "AQP4KO_sagital": "AQP4 KO\n3 mo", "Thy_CTRL": "WT\n12 mo"}
LBL1 = {k: v.replace("\n", " ") for k, v in LBL.items()}
PLANE = {"SPF_Coronar": "coronal", "AQP4KO_sagital": "sagittal", "Thy_CTRL": "sagittal"}
# orientation per section: (rotation in degrees counter-clockwise, flip left-right, flip up-down); set from cell K6a
ORIENT = globals().get("ORIENT", {"SPF_Coronar": (0, False, False), "AQP4KO_sagital": (180, False, False), "Thy_CTRL": (270, False, False)})

def orient(oo, s):
    """Rotate and flip section coordinates for display (statistics are not affected)."""
    rot, fx, fy = ORIENT[s]; oo = oo.copy()
    cx, cy = oo["x"].mean(), oo["y"].mean(); x, y = oo["x"] - cx, oo["y"] - cy
    t = np.deg2rad(rot); x, y = x * np.cos(t) + y * np.sin(t), -x * np.sin(t) + y * np.cos(t)   # y axis points down in the plots
    oo["x"], oo["y"] = (-x if fx else x) + cx, (-y if fy else y) + cy
    return oo

def wilson(k, n):
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson"); return 100 * k / n, 100 * lo, 100 * hi

ORD = [("SPF_Coronar", "#c9d1ee", "o"), ("AQP4KO_sagital", "#9aa8dc", "s"), ("Thy_CTRL", C_OLD, "D")]
COL = {s: c for s, c, _ in ORD}
FORE = R.get("FORE", ["Cortex", "Hippocampus", "Hypothalamus", "Pallidum", "Striatum", "Thalamus"])
MF = mg_all[mg_all["home_region"].isin(FORE)]
or_fore = {ref: glm_or(MF[MF["sample"].isin([ref, "Thy_CTRL"])], "spp1 ~ aged + ctx_bin + home_region", "aged")
           for ref in ["SPF_Coronar", "AQP4KO_sagital"]}
sig = lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."

def tissue_image(oo, res=40):
    """Rasterised tissue: blue-grey where fibre/border cells dominate a tile, light grey elsewhere."""
    xb = np.arange(oo["x"].min(), oo["x"].max() + res, res); yb = np.arange(oo["y"].min(), oo["y"].max() + res, res)
    H_all, _, _ = np.histogram2d(oo["x"], oo["y"], bins=[xb, yb])
    fbm = oo["ctx_bin"].isin(["Fibre", "Border"]).to_numpy()
    H_fb, _, _ = np.histogram2d(oo["x"][fbm], oo["y"][fbm], bins=[xb, yb])
    img = np.zeros(H_all.shape + (4,))
    img[H_all > 0] = to_rgba("#ececec")
    img[(H_all > 0) & (H_fb / np.maximum(H_all, 1) >= 0.5)] = to_rgba("#bfcde0")
    return img.transpose(1, 0, 2), (xb[0], xb[-1], yb[0], yb[-1])

# ================= sections (oriented) first, so the height of row B can follow the real section shapes
secs = []
for s, _, _ in ORD:
    oo = o_all[(o_all["sample"] == s) & (o_all["shared"] if s == "SPF_Coronar" else True)]
    secs.append((s, orient(oo, s)))
W_ = [(oo["x"].max() - oo["x"].min()) * 1.04 for _, oo in secs]
H_ = max(oo["y"].max() - oo["y"].min() for _, oo in secs) * 1.06
hB = H_ / sum(W_) * 180 * 0.85 * 0.97 + 2                            # mm needed by row B at full width (+ title)
HR = [30, hB, 50, 54]                                                 # row heights in mm: A, B, C-E, F-H
fig = plt.figure(figsize=(180 * MM, sum(HR) * (1 + 3 * 0.42 / 4) / 0.92 * MM))
gs = gridspec.GridSpec(4, 12, figure=fig, height_ratios=HR, hspace=0.42, wspace=3.0,
                       left=0.10, right=0.95, top=0.97, bottom=0.05)

# ================= A: study design on an age axis (left) and neighbourhood analysis (right)
gA = gs[0, :].subgridspec(1, 2, width_ratios=[1.5, 1], wspace=0.12)
a1 = fig.add_subplot(gA[0]); a1.axis("off"); a1.set_xlim(-0.5, 14.5); a1.set_ylim(-1.25, 1.25)
a1.plot([0, 13.5], [0, 0], color="#555555", lw=0.8)
for t_ in [0, 3, 6, 9, 12]:
    a1.plot([t_, t_], [-0.08, 0.08], color="#555555", lw=0.6)
    a1.text(t_, -0.3, str(t_), ha="center", va="top", fontsize=5.3)
a1.text(13.7, 0, "months", va="center", fontsize=5.3)
PTS = {"SPF_Coronar": (3, 0.62), "AQP4KO_sagital": (3, -0.95), "Thy_CTRL": (12, 0.62)}
for s, (x_, y_) in PTS.items():
    a1.plot([x_, x_], [0, y_ * 0.55], color="#bbbbbb", lw=0.6, ls=(0, (2, 2)), zorder=1)
    a1.scatter(x_, y_ * 0.55 if y_ > 0 else -0.55, s=0, zorder=0)
    a1.scatter(x_, 0, s=22, color=COL[s], edgecolor="white", lw=0.4, zorder=3)
    a1.text(x_, y_, f"$\\bf{{{LBL1[s].replace(' ', '~')}}}$\n{PLANE[s]}", ha="center", va="center", fontsize=5.6,
            bbox=dict(boxstyle="round,pad=0.25", fc=to_rgba(COL[s], 0.35), ec="none"))
a1.annotate("", xy=(10.6, 0.62), xytext=(4.4, 0.62), arrowprops=dict(arrowstyle="<->", lw=0.6, color="#666666"))
a1.text(7.5, 0.82, "matched forebrain (C)", ha="center", fontsize=5, color="#444444")
a1.annotate("", xy=(12, -0.12), xytext=(4.9, -0.95), arrowprops=dict(arrowstyle="<->", lw=0.6, color="#666666",
            connectionstyle="angle,angleA=0,angleB=90,rad=0"))
a1.text(8.2, -1.12, "same plane (D)", ha="center", fontsize=5, color="#444444")
a1.set_title("# S4A  Study design", loc="left", fontsize=7, fontweight="bold")

a2 = fig.add_subplot(gA[1]); a2.axis("off"); a2.set_xlim(0, 2.5); a2.set_ylim(0, 1.05)
a2.set_aspect("equal", adjustable="box"); a2.set_anchor("W")
rs = np.random.default_rng(3)
a2.scatter(rs.uniform(0.04, 0.70, 26), rs.uniform(0.08, 0.97, 26), s=6, c=C_FIB, lw=0)
P1, P2 = (0.24, 0.62), (0.50, 0.40)                                   # two Spp1+ microglia = one cluster
a2.add_patch(Circle(P1, 0.15, fill=False, ec="#777777", lw=0.6, ls=(0, (2, 2))))
a2.add_patch(Ellipse(((P1[0] + P2[0]) / 2, (P1[1] + P2[1]) / 2), 0.62, 0.36, angle=-40, fill=False, ec=C_POS, lw=0.8))
a2.scatter(*zip(P1, P2), s=40, c=C_POS, edgecolor="white", lw=0.5, zorder=4)
a2.scatter([0.33, 0.10, 0.62], [0.74, 0.20, 0.80], s=18, c=C_AS, lw=0, zorder=3)
for y_, kind, c_, t_ in [(0.92, "dot", C_AS, "astrocyte"), (0.70, "dot", C_POS, "Spp1+ microglia"),
                         (0.48, "dot", C_FIB, "white matter cell"), (0.26, "dash", "#777777", "25 µm neighbourhood"),
                         (0.06, "ring", C_POS, "cluster: Spp1+ microglia\nwithin 50 µm of each other")]:
    if kind == "dot": a2.scatter([0.92], [y_], s=12, c=c_, lw=0)
    elif kind == "dash": a2.add_patch(Circle((0.92, y_), 0.05, fill=False, ec=c_, lw=0.6, ls=(0, (2, 2))))
    else: a2.add_patch(Ellipse((0.92, y_), 0.14, 0.09, fill=False, ec=c_, lw=0.8))
    a2.text(1.02, y_, t_, fontsize=5.2, va="center")
a2.set_title("Neighbourhood analysis", loc="left", fontsize=6.5)

# ================= B: three sections, full width, common scale (panel widths proportional to section widths)
gB = gs[1, :].subgridspec(1, 3, width_ratios=W_, wspace=0.06)
for k, (s, oo) in enumerate(secs):
    ax = fig.add_subplot(gB[k])
    img, ext = tissue_image(oo)
    ax.imshow(img, extent=ext, origin="lower", interpolation="nearest", zorder=1)
    pp = oo[oo["is_mg"] & (oo["Spp1_n"] >= 1)]; fb = pp["ctx_bin"].isin(["Fibre", "Border"])
    ax.scatter(pp.loc[~fb, "x"], pp.loc[~fb, "y"], s=4, c="#e9b7cb", lw=0, zorder=4)
    ax.scatter(pp.loc[fb, "x"], pp.loc[fb, "y"], s=8, c=C_POS, lw=0.25, edgecolor="white", zorder=5)
    cxs, cys = (oo["x"].min() + oo["x"].max()) / 2, (oo["y"].min() + oo["y"].max()) / 2
    ax.set_xlim(cxs - W_[k] / 2, cxs + W_[k] / 2); ax.set_ylim(cys + H_ / 2, cys - H_ / 2)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_anchor("N")        # common scale, top-aligned titles
    x1 = cxs + W_[k] / 2
    ax.plot([x1 - 1100, x1 - 100], [cys + H_ / 2 - 150] * 2, color="#222222", lw=1.1)
    ax.text(x1 - 600, cys + H_ / 2 - 380, "1 mm", ha="center", fontsize=5)
    ax.set_title(("# S4B  " if k == 0 else "") + f"{LBL1[s]}, {PLANE[s]}", loc="left", fontsize=7, fontweight="bold" if k == 0 else "normal")

# ================= C: matched forebrain, overall Spp1+ share per animal
axC = fig.add_subplot(gs[2, 0:3])
for i, (s, c_, _) in enumerate(ORD):
    t = MF[MF["sample"] == s]; p_, lo_, hi_ = wilson(int(t["spp1"].sum()), len(t))
    axC.bar(i, p_, width=0.7, color=c_)
    axC.errorbar(i, p_, yerr=[[p_ - lo_], [hi_ - p_]], color="#333333", lw=0.8, capsize=2.5)
axC.set_xticks(range(3)); axC.set_xticklabels([LBL[s] for s, _, _ in ORD], fontsize=5.8)
axC.set_ylabel("Spp1+ microglia (%)")
yt = max(axC.get_ylim()[1], 4.0)
for i, ref in enumerate(["SPF_Coronar", "AQP4KO_sagital"]):
    h = yt * (1.08 + 0.16 * (1 - i))
    axC.plot([i, i, 2, 2], [h - yt * 0.03, h, h, h - yt * 0.03], color="#333333", lw=0.6)
    axC.text((i + 2) / 2, h + yt * 0.01, sig(or_fore[ref][3]), ha="center", fontsize=6)
axC.set_ylim(0, yt * 1.45)
axC.set_title("# S4C  Matched forebrain", loc="left", fontsize=7, fontweight="bold")

# ================= D: same plane, by home region
axD = fig.add_subplot(gs[2, 4:8])
rb = R["regB"]; lk, la = SAMPLES["AQP4KO_sagital"][0], SAMPLES["Thy_CTRL"][0]
rb = rb[(rb[("size", lk)] >= 100) & (rb[("size", la)] >= 100)]
pk, pa = 100 * rb[("sum", lk)] / rb[("size", lk)], 100 * rb[("sum", la)] / rb[("size", la)]
order = pa.sort_values().index; yy = np.arange(len(order))
axD.barh(yy - 0.2, pk.loc[order], height=0.38, color=COL["AQP4KO_sagital"], label=LBL1["AQP4KO_sagital"])
axD.barh(yy + 0.2, pa.loc[order], height=0.38, color=C_OLD, label=LBL1["Thy_CTRL"])
axD.set_yticks(yy); axD.set_yticklabels(order, fontsize=6); axD.set_xlabel("Spp1+ microglia (%)")
axD.legend(fontsize=5.3, loc="lower right")
axD.set_title("# S4D  Same plane, by region", loc="left", fontsize=7, fontweight="bold")

# ================= E: clustering
axE = fig.add_subplot(gs[2, 9:12])
for i, (s, c_, _) in enumerate(ORD):
    obs_c, null, pc = R["clus"][s]
    axE.scatter(np.full(len(null), i) + np.random.default_rng(1).uniform(-0.15, 0.15, len(null)), 100 * null,
                s=2, color="#d5d5d5", lw=0, rasterized=True)
    axE.scatter(i, 100 * obs_c, s=34, marker="D", color=C_POS, zorder=3)
    axE.text(i + 0.22, 100 * obs_c, sig(pc), va="center", ha="left", fontsize=6 if pc < 0.05 else 5)
axE.set_xticks(range(3)); axE.set_xticklabels([LBL[s] for s, _, _ in ORD], fontsize=5.8)
axE.set_ylabel("Clustered Spp1+ (%)")
axE.scatter([], [], s=30, marker="D", color=C_POS, label="observed"); axE.scatter([], [], s=6, color="#d5d5d5", label="permuted")
axE.legend(fontsize=5, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False)
axE.set_ylim(0, max(max(100 * R["clus"][s][0], 100 * np.max(R["clus"][s][1])) for s, _, _ in ORD) * 1.35 + 3)
axE.set_title("# S4E  Clustering", loc="left", fontsize=7, fontweight="bold")

# ================= F: zoom
axF = fig.add_subplot(gs[3, 0:3])
oa = orient(o_all[o_all["sample"] == "Thy_CTRL"], "Thy_CTRL")
pf = oa[oa["is_mg"] & (oa["Spp1_n"] >= 1) & oa["ctx_bin"].isin(["Fibre", "Border"])]
pxy = pf[["x", "y"]].to_numpy(); cx, cy = pxy[cKDTree(pxy).query_ball_point(pxy, 120, return_length=True).argmax()]
WZ = 240
z = oa[oa["x"].between(cx - WZ / 2, cx + WZ / 2) & oa["y"].between(cy - WZ / 2, cy + WZ / 2)]
axF.scatter(z["x"], z["y"], s=5, c=np.where(z["cell_type"] == "Oligodendrocyte", C_FIB, "#e6e6e6"), lw=0)
za = z[z["is_astro"]]; axF.scatter(za["x"], za["y"], s=12, c=C_AS, lw=0, zorder=3)
zm = z[z["is_mg"]]; axF.scatter(zm["x"], zm["y"], s=12, c=C_MG, lw=0, zorder=4)
zp = zm[zm["Spp1_n"] >= 1]; axF.scatter(zp["x"], zp["y"], s=28, c=C_POS, lw=0.5, edgecolor="white", zorder=5)
for xx_, yy_ in zp[["x", "y"]].to_numpy():
    axF.add_patch(Circle((xx_, yy_), 25, fill=False, ec="#888888", lw=0.4, ls=(0, (2, 2)), zorder=2))
axF.set_xlim(cx - WZ / 2, cx + WZ / 2); axF.set_ylim(cy + WZ / 2, cy - WZ / 2); axF.set_aspect("equal")
axF.set_xticks([]); axF.set_yticks([])
for s_ in axF.spines.values(): s_.set_visible(True); s_.set_linewidth(0.6)
axF.plot([cx + WZ / 2 - 60, cx + WZ / 2 - 10], [cy + WZ / 2 - 12] * 2, color="#222222", lw=1.1)
axF.text(cx + WZ / 2 - 35, cy + WZ / 2 - 25, "50 µm", ha="center", fontsize=5)
for lab_, c_ in [("Oligodendrocyte", C_FIB), ("Astrocyte", C_AS), ("Microglia", C_MG), ("Spp1+ microglia", C_POS)]:
    axF.scatter([], [], s=10, c=c_, label=lab_)
axF.legend(loc="upper left", bbox_to_anchor=(-0.05, -0.02), ncol=2, fontsize=4.9, handletextpad=0.2, columnspacing=0.4)
axF.set_title("# S4F  Spp1+ cluster, WT 12 mo", loc="left", fontsize=7, fontweight="bold")

# ================= G: astrocyte contact (fibre/border)
axG = fig.add_subplot(gs[3, 4:8])
for i, (s, c_, _) in enumerate(ORD):
    Y = mg_all[(mg_all["sample"] == s) & (mg_all["shared"] if s == "SPF_Coronar" else True) & mg_all["ctx_bin"].isin(["Fibre", "Border"])]
    cont = Y["n_astro25"] >= 1; vals = []
    for j, sp_ in enumerate([0, 1]):
        sel = Y["spp1"] == sp_; v, lo_, hi_ = wilson(int(cont[sel].sum()), int(sel.sum())); vals.append(hi_)
        axG.bar(i * 2.6 + j, v, width=0.85, color=c_ if sp_ else "white", edgecolor=c_, lw=1.0)
        axG.errorbar(i * 2.6 + j, v, yerr=[[v - lo_], [hi_ - v]], color="#333333", lw=0.8, capsize=2.5)
    a = int((cont & (Y["spp1"] == 1)).sum()); b = int((Y["spp1"] == 1).sum()) - a
    c = int((cont & (Y["spp1"] == 0)).sum()); d = int((Y["spp1"] == 0).sum()) - c
    p = fisher_exact([[a, b], [c, d]])[1]; top = max(vals) + 6
    axG.plot([i * 2.6, i * 2.6, i * 2.6 + 1, i * 2.6 + 1], [top - 2, top, top, top - 2], color="#333333", lw=0.6)
    axG.text(i * 2.6 + 0.5, top + 1, sig(p), ha="center", fontsize=6 if p < 0.05 else 5)
axG.set_xticks([i * 2.6 + 0.5 for i in range(3)])
axG.set_xticklabels([LBL[s] for s, _, _ in ORD], fontsize=5.8)
axG.legend(handles=[Patch(facecolor="white", edgecolor="#555555", lw=0.8, label="Spp1−"),
                    Patch(facecolor="#555555", edgecolor="#555555", lw=0.8, label="Spp1+")],
           fontsize=5.3, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False)
axG.set_ylim(0, 100); axG.set_ylabel("Microglia with\nastrocyte contact (%)")
axG.set_title("# S4G  Astrocyte contact", loc="left", fontsize=7, fontweight="bold")

# ================= H: dot plot (values from Z2: context x home-region strata)
CM_DOT = LinearSegmentedColormap.from_list("dot", ["#f1f3fb", "#9aa8dc", "#2b3f9e"])
axH = fig.add_subplot(gs[3, 9:12])
z2_ = pd.read_csv(TAB / "Z2_astro_near_spp1_shell_radius_controls.csv")
z2_ = z2_[z2_["config"] == "0-25 µm"]
genes_ = ["Gfap", "Aqp4", "Cd44"]; rows = []
for s, _, _ in ORD:
    rows += [(f"{LBL1[s]} · Spp1−", s, "neg"), (f"{LBL1[s]} · Spp1+", s, "pos")]

def zrow(s, g):
    return z2_[(z2_["sample"] == SAMPLES[s][0]) & (z2_["gene"] == g)].iloc[0]

means = np.array([[zrow(s, g)["expr_near_pos" if w_ == "pos" else "expr_expected"] for g in genes_] for _, s, w_ in rows])
fracs = np.array([[zrow(s, g)["det_near_pos" if w_ == "pos" else "det_expected"] for g in genes_] for _, s, w_ in rows])
scaled = (means - means.min(0)) / np.maximum(means.max(0) - means.min(0), 1e-9)
for i in range(len(rows)):
    for j in range(len(genes_)):
        axH.scatter(j, i, s=6 + 1.4 * fracs[i, j], c=[CM_DOT(scaled[i, j])], edgecolor="#555555", lw=0.3, zorder=3)
        if rows[i][2] == "pos":
            lab = sig(zrow(rows[i][1], genes_[j])["p_expr"])
            if lab != "n.s.": axH.text(j + 0.32, i, lab, va="center", fontsize=6)
for b in [1.5, 3.5]: axH.axhline(b, color="#dddddd", lw=0.6)
axH.set_yticks(range(len(rows))); axH.set_yticklabels([r[0] for r in rows], fontsize=5.2)
axH.set_xticks(range(len(genes_))); axH.set_xticklabels(genes_, style="italic", fontsize=6)
axH.set_xlim(-0.6, len(genes_) - 0.1); axH.set_ylim(len(rows) - 0.5, -0.5)
for s_ in axH.spines.values(): s_.set_visible(False)
axH.tick_params(length=0)
for v in [25, 50, 75]:
    axH.scatter([], [], s=6 + 1.4 * v, c="#cccccc", edgecolor="#555555", lw=0.3, label=f"{v}%")
axH.legend(fontsize=4.8, loc="upper left", bbox_to_anchor=(1.0, 1.02), labelspacing=1.0, handletextpad=0.2)
cax = axH.inset_axes([1.12, 0.0, 0.07, 0.3])
cb = plt.colorbar(plt.cm.ScalarMappable(cmap=CM_DOT), cax=cax); cb.set_ticks([0, 1]); cb.set_ticklabels(["low", "high"])
cb.ax.tick_params(labelsize=4.8); cb.outline.set_visible(False)
axH.set_title("# S4H  Astrocyte neighbours", loc="left", fontsize=7, fontweight="bold")

save(fig, "FigS4")

In [ ]:
# ==========================================================
# K8) Figure 2: midlife validation in our WT 12 mo mouse (uses objects from K6, run K6 first)
# ==========================================================
S_OLD = "Thy_CTRL"
fig = plt.figure(figsize=(180 * MM, 150 * MM))
gs2 = gridspec.GridSpec(2, 12, figure=fig, height_ratios=[1.15, 0.9], hspace=0.5, wspace=3.0,
                        left=0.09, right=0.95, top=0.96, bottom=0.08)

# ================= A: WT 12 mo section at single-cell resolution, white matter and Spp1+ microglia
C_WMF, C_WMB, C_GREY = "#6f8fb3", "#b4c6dc", "#f0f0f0"           # white matter stronger, grey matter as background
axA = fig.add_subplot(gs2[0, 0:8])
oo = dict(secs)[S_OLD]
for ctx_, c_, s_ in [("Grey", C_GREY, 0.15), ("Border", C_WMB, 0.6), ("Fibre", C_WMF, 0.8)]:   # grey first, white matter on top
    t_ = oo[oo["ctx_bin"] == ctx_]
    axA.scatter(t_["x"], t_["y"], s=s_, c=c_, lw=0, rasterized=True, zorder=1)
mm_ = oo[oo["is_mg"]]
axA.scatter(mm_["x"], mm_["y"], s=0.4, c=C_MG, lw=0, alpha=0.7, rasterized=True, zorder=2)  # all microglia            # every cell
mm_ = oo[oo["is_mg"]]
axA.scatter(mm_["x"], mm_["y"], s=0.6, c=C_MG, lw=0, rasterized=True, zorder=2)            # all microglia
pp = mm_[mm_["Spp1_n"] >= 1]; fb = pp["ctx_bin"].isin(["Fibre", "Border"])
axA.scatter(pp.loc[~fb, "x"], pp.loc[~fb, "y"], s=5, c="#e9b7cb", lw=0, zorder=4)
axA.scatter(pp.loc[fb, "x"], pp.loc[fb, "y"], s=10, c=C_POS, lw=0.3, edgecolor="white", zorder=5)
axA.set_xlim(oo["x"].min() - 100, oo["x"].max() + 100); axA.set_ylim(oo["y"].max() + 100, oo["y"].min() - 100)
axA.set_aspect("equal"); axA.axis("off")
x1, y1 = oo["x"].max(), oo["y"].max()
axA.plot([x1 - 1000, x1], [y1 + 20] * 2, color="#222222", lw=1.1); axA.text(x1 - 500, y1 - 120, "1 mm", ha="center", fontsize=5)
for lab_, c_ in [("White matter", C_WMF), ("White matter border", C_WMB), ("Microglia", C_MG),
                 ("Spp1+ microglia, white matter", C_POS), ("Spp1+ microglia, grey matter", "#e9b7cb")]:
    axA.scatter([], [], s=14, c=c_, label=lab_)
axA.legend(loc="upper left", bbox_to_anchor=(0.0, -0.01), ncol=3, fontsize=5.3, frameon=False, handletextpad=0.3)
axA.set_title("# 2A  WT 12 mo, sagittal", loc="left", fontsize=7, fontweight="bold")

# ================= B: zoom on a cluster (same window as Supplementary Figure S4F)
axB = fig.add_subplot(gs2[0, 8:12])
axB.scatter(z["x"], z["y"], s=5, c=np.where(z["cell_type"] == "Oligodendrocyte", C_FIB, "#e6e6e6"), lw=0)
axB.scatter(za["x"], za["y"], s=12, c=C_AS, lw=0, zorder=3)
axB.scatter(zm["x"], zm["y"], s=12, c=C_MG, lw=0, zorder=4)
axB.scatter(zp["x"], zp["y"], s=28, c=C_POS, lw=0.5, edgecolor="white", zorder=5)
for xx_, yy_ in zp[["x", "y"]].to_numpy():
    axB.add_patch(Circle((xx_, yy_), 25, fill=False, ec="#888888", lw=0.4, ls=(0, (2, 2)), zorder=2))
axB.set_xlim(cx - WZ / 2, cx + WZ / 2); axB.set_ylim(cy + WZ / 2, cy - WZ / 2); axB.set_aspect("equal")
axB.set_xticks([]); axB.set_yticks([])
for s_ in axB.spines.values(): s_.set_visible(True); s_.set_linewidth(0.6)
axB.plot([cx + WZ / 2 - 60, cx + WZ / 2 - 10], [cy + WZ / 2 - 12] * 2, color="#222222", lw=1.1)
axB.text(cx + WZ / 2 - 35, cy + WZ / 2 - 25, "50 µm", ha="center", fontsize=5)
for lab_, c_ in [("Oligodendrocyte", C_FIB), ("Astrocyte", C_AS), ("Microglia", C_MG), ("Spp1+ microglia", C_POS)]:
    axB.scatter([], [], s=10, c=c_, label=lab_)
axB.legend(loc="upper left", bbox_to_anchor=(-0.05, -0.02), ncol=2, fontsize=4.9, handletextpad=0.2, columnspacing=0.4, frameon=False)
axB.set_title("# 2B  Spp1+ cluster", loc="left", fontsize=7, fontweight="bold")

# ================= C: clustering in the WT 12 mo mouse
axC2 = fig.add_subplot(gs2[1, 0:3])
obs_c, null, pc = R["clus"][S_OLD]; null = np.asarray(null, float)
axC2.scatter(np.random.default_rng(1).uniform(-0.15, 0.15, len(null)), 100 * null, s=2, color="#d5d5d5", lw=0, rasterized=True)
axC2.scatter(0, 100 * obs_c, s=40, marker="D", color=C_POS, zorder=3)
axC2.text(0.22, 100 * obs_c, sig(pc), va="center", fontsize=6)
axC2.set_xticks([0]); axC2.set_xticklabels(["WT\n12 mo"], fontsize=5.8); axC2.set_xlim(-0.6, 0.8)
axC2.set_ylim(0, max(100 * obs_c, 100 * null.max()) * 1.3 + 3); axC2.set_ylabel("Clustered Spp1+ (%)")
axC2.scatter([], [], s=30, marker="D", color=C_POS, label="observed"); axC2.scatter([], [], s=6, color="#d5d5d5", label="permuted")
axC2.legend(fontsize=5, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False)
axC2.set_title("# 2C  Clustering", loc="left", fontsize=7, fontweight="bold")

# ================= D: astrocyte contact in and around white matter (WT 12 mo)
axD2 = fig.add_subplot(gs2[1, 4:7])
Y = mg_all[(mg_all["sample"] == S_OLD) & mg_all["ctx_bin"].isin(["Fibre", "Border"])]
cont = Y["n_astro25"] >= 1; vals = []
for j, sp_ in enumerate([0, 1]):
    sel = Y["spp1"] == sp_; v, lo_, hi_ = wilson(int(cont[sel].sum()), int(sel.sum())); vals.append(hi_)
    axD2.bar(j, v, width=0.75, color=C_OLD if sp_ else "white", edgecolor=C_OLD, lw=1.0)
    axD2.errorbar(j, v, yerr=[[v - lo_], [hi_ - v]], color="#333333", lw=0.8, capsize=2.5)
a = int((cont & (Y["spp1"] == 1)).sum()); b = int((Y["spp1"] == 1).sum()) - a
c = int((cont & (Y["spp1"] == 0)).sum()); d = int((Y["spp1"] == 0).sum()) - c
p = fisher_exact([[a, b], [c, d]])[1]; top = max(vals) + 6
axD2.plot([0, 0, 1, 1], [top - 2, top, top, top - 2], color="#333333", lw=0.6); axD2.text(0.5, top + 1, sig(p), ha="center", fontsize=6)
axD2.set_xticks([0, 1]); axD2.set_xticklabels(["Spp1−", "Spp1+"]); axD2.set_xlim(-0.6, 1.6)
axD2.set_ylim(0, 100); axD2.set_ylabel("Microglia with\nastrocyte contact (%)")
axD2.set_title("# 2D  Astrocyte contact", loc="left", fontsize=7, fontweight="bold")

# ================= E: astrocyte neighbours (WT 12 mo, values from Z2 as in Supplementary Figure S4H)
axE2 = fig.add_subplot(gs2[1, 8:12])
rows2 = [("Spp1−", S_OLD, "neg"), ("Spp1+", S_OLD, "pos")]
genes2 = ["Gfap", "Aqp4", "Cd44"]
m2 = np.array([[zrow(s, g)["expr_near_pos" if w_ == "pos" else "expr_expected"] for g in genes2] for _, s, w_ in rows2])
f2_ = np.array([[zrow(s, g)["det_near_pos" if w_ == "pos" else "det_expected"] for g in genes2] for _, s, w_ in rows2])
sc2 = (m2 - m2.min(0)) / np.maximum(m2.max(0) - m2.min(0), 1e-9)
for i in range(2):
    for j in range(3):
        axE2.scatter(j, i, s=10 + 2.2 * f2_[i, j], c=[CM_DOT(sc2[i, j])], edgecolor="#555555", lw=0.3, zorder=3)
        if i == 1:
            lab = sig(zrow(S_OLD, genes2[j])["p_expr"])
            if lab != "n.s.": axE2.text(j + 0.3, i, lab, va="center", fontsize=6)
axE2.set_yticks([0, 1]); axE2.set_yticklabels(["near Spp1−", "near Spp1+"], fontsize=6)
axE2.set_xticks(range(3)); axE2.set_xticklabels(genes2, style="italic", fontsize=6)
axE2.set_xlim(-0.6, 2.6); axE2.set_ylim(1.6, -0.6)
for s_ in axE2.spines.values(): s_.set_visible(False)
axE2.tick_params(length=0)
for v in [25, 50, 75]:
    axE2.scatter([], [], s=10 + 2.2 * v, c="#cccccc", edgecolor="#555555", lw=0.3, label=f"{v}%")
axE2.legend(fontsize=4.8, loc="upper center", bbox_to_anchor=(0.45, -0.18), ncol=3, frameon=False, columnspacing=0.8)
axE2.set_title("# 2E  Astrocytes near microglia", loc="left", fontsize=7, fontweight="bold")
save(fig, "Fig2")

In [ ]:
# ==========================================================
# K7) Supplementary Table S2: source data of Figure 2 and Supplementary Figure S4
# ==========================================================
from openpyxl.styles import Font
XLS = OUT / "final_data" / "supplementary_tables"; XLS.mkdir(parents=True, exist_ok=True)


# old sample labels (SAMPLES, Z tables) -> display labels in the Excel sheets
REN = {"SPF 12 wk": "WT 3 mo", "AQP4 KO 12 wk": "AQP4 KO 3 mo", "Healthy 1 y": "WT 12 mo", "healthy 1 y": "WT 12 mo",
       "1-year brain": "WT 12 mo section", "12 weeks": "3 months", "1 year": "12 months"}
def relabel(x):
    """Replace old sample labels in a string or in a DataFrame (cells, column names, index)."""
    if isinstance(x, pd.DataFrame):
        x = x.rename(columns=relabel, index=relabel)
        return x.apply(lambda c: c if pd.api.types.is_numeric_dtype(c) else c.map(relabel))
    if isinstance(x, str):
        for a, b in REN.items(): x = x.replace(a, b)
    return x


def write_book(path, title, sheets):
    """Excel workbook: README sheet (title + list of sheets), then one sheet per panel in the given order."""
    readme = pd.DataFrame([(n, relabel(d), len(df)) for n, d, df in sheets], columns=["Sheet", "Content", "Rows"])
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        readme.to_excel(xw, sheet_name="README", index=False, startrow=2)
        for n, d, df in sheets:
            relabel(df).to_excel(xw, sheet_name=str(n)[:31], index=False, startrow=2)
        wb = xw.book
        wb["README"]["A1"] = title; wb["README"]["A1"].font = Font(bold=True, size=12)
        for n, d, df in sheets:
            ws = wb[str(n)[:31]]; ws["A1"] = relabel(d); ws["A1"].font = Font(bold=True)
        for ws in wb.worksheets:
            for c in ws[3]: c.font = Font(bold=True)
            ws.freeze_panes = "A4"
            for col in ws.iter_cols(min_row=3, max_row=min(ws.max_row, 200)):
                w = max(len(str(c.value)) if c.value is not None else 0 for c in col)
                ws.column_dimensions[col[0].column_letter].width = min(max(9, w + 2), 60)
        wb["README"].column_dimensions["B"].width = 110
    print("written:", path.name, "|", ", ".join(n for n, _, _ in sheets))


lab_ = {s: SAMPLES[s][0] for s, _, _ in ORD}
cols_ = lambda d, cs: d[[c for c in cs if c in d.columns]]

# A: design
sA = []
for s, _, _ in ORD:
    oo = o_all[o_all["sample"] == s]
    sA.append({"section": lab_[s], "sample id": s, "plane": "coronal" if s == "SPF_Coronar" else "sagittal",
               "age": "1 year" if s == "Thy_CTRL" else "12 weeks", "genotype": "AQP4-deficient" if s == "AQP4KO_sagital" else "wild type",
               "cells": len(oo), "microglia": int(oo["is_mg"].sum()), "astrocytes": int(oo["is_astro"].sum()) if "is_astro" in oo else np.nan})
sA = pd.DataFrame(sA)
# B: Spp1+ microglia shown on the sections
sB = []
for s, _, _ in ORD:
    oo = o_all[(o_all["sample"] == s) & (o_all["shared"] if s == "SPF_Coronar" else True)]
    pp = oo[oo["is_mg"] & (oo["Spp1_n"] >= 1)]
    sB.append(cols_(pp, ["x", "y", "ctx_bin", "Spp1_n"]).assign(section=lab_[s]))
sB = pd.concat(sB)[["section", "x", "y", "ctx_bin", "Spp1_n"]].rename(columns={"x": "x_um", "y": "y_um", "ctx_bin": "local context"})
# C: matched forebrain
sC = MF.groupby("sample").agg(microglia=("spp1", "size"), spp1_pos=("spp1", "sum")).reindex([s for s, _, _ in ORD])
sC["pct_spp1_pos"] = 100 * sC["spp1_pos"] / sC["microglia"]; sC.index = [lab_[s] for s in sC.index]
sC = sC.reset_index(names="section")
sC_m = pd.DataFrame([{"comparison": f"Healthy 1 y vs {lab_[ref]}", "OR": list(or_fore[ref])[0], "CI_lo": list(or_fore[ref])[1],
                      "CI_hi": list(or_fore[ref])[2], "p": list(or_fore[ref])[3],
                      "model": "binomial GLM: spp1 ~ aged + context + home region, HC1", "home regions": ", ".join(FORE)}
                     for ref in ["SPF_Coronar", "AQP4KO_sagital"]])
# D: same plane, by home region (as plotted)
sD = pd.DataFrame({"home region": order,
                   f"{lk} microglia": rb.loc[order, ("size", lk)].values, f"{lk} Spp1+": rb.loc[order, ("sum", lk)].values,
                   f"{lk} Spp1+ (%)": pk.loc[order].values,
                   f"{la} microglia": rb.loc[order, ("size", la)].values, f"{la} Spp1+": rb.loc[order, ("sum", la)].values,
                   f"{la} Spp1+ (%)": pa.loc[order].values})
# E: clustering
sE, sE_null = [], {}
for s, _, _ in ORD:
    obs_c, null, pc = R["clus"][s]; null = np.asarray(null, float)
    sE.append({"section": lab_[s], "observed (% clustered)": 100 * obs_c, "permuted mean (%)": 100 * np.nanmean(null),
               "permuted 2.5%": 100 * np.nanpercentile(null, 2.5), "permuted 97.5%": 100 * np.nanpercentile(null, 97.5),
               "permutations": len(null), "p": pc})
    sE_null[lab_[s]] = 100 * null
sE = pd.DataFrame(sE); sE_null = pd.DataFrame({k: pd.Series(v) for k, v in sE_null.items()})
# F: zoom window
sF = cols_(z, ["x", "y", "cell_type", "is_mg", "is_astro", "Spp1_n", "ctx_bin"]).rename(columns={"x": "x_um", "y": "y_um"})
sF.insert(0, "window", f"centre {cx:.0f}, {cy:.0f} µm, width {WZ} µm")
# G: astrocyte contact, border + fibre context
sG = []
for s, _, _ in ORD:
    Y = mg_all[(mg_all["sample"] == s) & (mg_all["shared"] if s == "SPF_Coronar" else True) & mg_all["ctx_bin"].isin(["Fibre", "Border"])]
    cont = Y["n_astro25"] >= 1
    a = int((cont & (Y["spp1"] == 1)).sum()); b = int((Y["spp1"] == 1).sum()) - a
    c = int((cont & (Y["spp1"] == 0)).sum()); d = int((Y["spp1"] == 0).sum()) - c
    sG.append({"section": lab_[s], "Spp1+ with astrocyte": a, "Spp1+ without": b, "Spp1+ contact (%)": 100 * a / max(a + b, 1),
               "Spp1− with astrocyte": c, "Spp1− without": d, "Spp1− contact (%)": 100 * c / max(c + d, 1),
               "Fisher p": fisher_exact([[a, b], [c, d]])[1]})
sG = pd.DataFrame(sG)
# H: astrocyte neighbours (Z2, 0 to 25 µm)
sH = z2_[z2_["gene"].isin(genes_)].copy()

write_book(XLS / "Supplementary_Table_S2_source_data_Figure2_FigureS4.xlsx", "Supplementary Table S2. Source data of Figure 2 (WT 12 mo) and Supplementary Figure S4 (all three mice)",
           [("S4A", "S4A: sections of our cohort (one mouse per condition)", sA),
            ("2A_S4B", "2A and S4B: Spp1+ microglia shown on the sections (coordinates in µm, local context)", sB),
            ("S4C", "S4C: Spp1+ microglia within forebrain home regions present in all three sections", sC),
            ("S4C_model", "S4C: age effect, binomial GLM with context and home region as fixed effects", sC_m),
            ("S4D", "S4D: Spp1+ microglia per home region in the two sagittal sections (regions with ≥100 microglia in both)", sD),
            ("2C_S4E", "2C and S4E: clustering, share of Spp1+ microglia with another Spp1+ microglia within 50 µm vs permutations", sE),
            ("2C_S4E_permutations", "2C and S4E: permuted values (%) per section", sE_null),
            ("2B_S4F", "2B and S4F: cells in the zoom window of the WT 12 mo brain", sF),
            ("2D_S4G", "2D and S4G: microglia with ≥1 astrocyte within 25 µm, border and fibre context, Fisher's exact test", sG),
            ("2E_S4H", "2E and S4H: astrocytes within 25 µm of Spp1+ vs Spp1− microglia (expected = Spp1− reference weighted to context x home-region strata)", sH)])